In [ ]:
import os
import sqlite3
import logging
from datetime import datetime
from pathlib import Path
from dotenv import load_dotenv

# Import your custom library
# Ensure loan_processor.py is in the same directory as this notebook
try:
    from loan_processor import LoanProcessor
except ImportError:
    print("⚠️ Error: loan_processor.py not found in the current directory.")

# Load environment variables (Ensure .env contains GOOGLE_AI_STUDIO)
load_dotenv()

# --- Configuration ---
# Use raw string (r"...") for Windows paths
ONEDRIVE_ROOT = Path(r"C:\Users\lopez\OneDrive\Documents\Bonanza Neiva\OneDrive\Clientes") 
SEARCH_KEYWORD = "credito ordinario"
DB_PATH = "processing_history.db"
OUTPUT_JSON_FOLDER = "client_json_data"

# The LoanProcessor library configures logging on import, 
# but we can add a handler to print to the notebook output if desired.
logging.getLogger().setLevel(logging.INFO)


In [ ]:
def init_db():
    """Initialize the tracking database if it doesn't exist."""
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    # We track file path, modification time, processed date, and status
    c.execute('''CREATE TABLE IF NOT EXISTS file_history
                 (file_path TEXT PRIMARY KEY, 
                  last_modified REAL, 
                  processed_at TEXT, 
                  status TEXT)''')
    conn.commit()
    return conn

def should_process(conn, file_path):
    """
    Decides if a file needs processing.
    Returns True if file is new, modified since last run, or previously failed.
    """
    try:
        current_mtime = file_path.stat().st_mtime
        c = conn.cursor()
        c.execute("SELECT last_modified, status FROM file_history WHERE file_path=?", (str(file_path),))
        row = c.fetchone()
        
        if row is None:
            return True # New file
            
        last_recorded_mtime, status = row
        
        # Process if file was modified since last record, or previous attempt failed
        if current_mtime > last_recorded_mtime or status != 'SUCCESS':
            return True
            
        return False # Already processed and unchanged
    except FileNotFoundError:
        return False

def mark_completed(conn, file_path, status="SUCCESS"):
    """Updates the database record for a file."""
    try:
        current_mtime = file_path.stat().st_mtime
        c = conn.cursor()
        c.execute('''INSERT OR REPLACE INTO file_history 
                     (file_path, last_modified, processed_at, status) 
                     VALUES (?, ?, ?, ?)''', 
                  (str(file_path), current_mtime, datetime.now().isoformat(), status))
        conn.commit()
    except Exception as e:
        print(f"⚠️ Error updating DB for {file_path.name}: {e}")


In [ ]:
def find_target_files(root_path, keyword, silent=False):
    """
    Generator that yields paths to PDF files containing the keyword.
    """
    root = Path(root_path)
    if not root.exists():
        print(f"❌ Error: Path not found: {root}")
        return

    if not silent:
        print(f"📂 Scanning {root}...")
    
    # rglob recursively finds files. We filter for .pdf and the keyword.
    for path in root.rglob("*.pdf"):
        if keyword.lower() in path.name.lower():
            yield path



In [ ]:
def count_pending_files(conn, root_path, keyword):
    """
    Helper function to count how many files actually need processing
    before starting the main loop.
    """
    print("🔍 Analyzing workload (counting pending files)...")
    count = 0
    # We use silent=True to avoid printing "Scanning..." twice
    for file_path in find_target_files(root_path, keyword, silent=True):
        if should_process(conn, file_path):
            count += 1
    return count

In [ ]:
conn = init_db()
count_pending_files(conn, ONEDRIVE_ROOT, SEARCH_KEYWORD)

In [ ]:
def run_batch_process():
    # 1. Initialize the LoanProcessor
    try:
        processor = LoanProcessor() 
        print("✅ LoanProcessor initialized successfully.")
    except Exception as e:
        print(f"❌ Failed to initialize LoanProcessor: {e}")
        return

    conn = init_db()
    
    try:
        # 2. Count files first
        total_pending = count_pending_files(conn, ONEDRIVE_ROOT, SEARCH_KEYWORD)
        
        if total_pending == 0:
            print("✅ No new files to process. Everything is up to date!")
            return

        print(f"📊 Found {total_pending} files waiting to be processed.")
        print(f"🚀 Starting batch process...")
        print(f"📂 Output folder: {Path(OUTPUT_JSON_FOLDER).absolute()}")
        
        processed_count = 0
        errors_count = 0
        
        # 3. Iterate through files found by the scanner
        for file_path in find_target_files(ONEDRIVE_ROOT, SEARCH_KEYWORD):
            
            if should_process(conn, file_path):
                processed_count += 1
                print(f"⚡ Processing [{processed_count}/{total_pending}]: {file_path.name}")
                
                # --- CALL YOUR LIBRARY HERE ---
                result = processor.process_single_file(
                    pdf_path=str(file_path), 
                    output_folder=OUTPUT_JSON_FOLDER
                )
                # ------------------------------
                
                if result:
                    print(f"   ✅ Extracted data.")
                    mark_completed(conn, file_path, status="SUCCESS")
                else:
                    print(f"   ❌ Extraction failed (check logs).")
                    mark_completed(conn, file_path, status="ERROR")
                    errors_count += 1
                
    except KeyboardInterrupt:
        print("\n🛑 Process interrupted by user.")
    finally:
        conn.close()
        
    print("\n" + "="*30)
    print(f"✅ Batch Complete")
    print(f"🆕 Processed: {processed_count}/{total_pending}")
    print(f"❌ Errors:    {errors_count}")
    print("="*30)

# Execute
run_batch_process()
